In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [42]:
from __future__ import print_function, division
import os
import glob
import torch
import torch.nn as nn
import pandas as pd
from skimage import io, transform
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils, datasets, models

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")
# os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
# os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [ ]:
!mkdir -p /content/local/

!mkdir -p /content/local/train
!mkdir -p /content/local/val
!mkdir -p /content/local/test

!echo "Unzipping test & val sets"
!unzip -q "/content/drive/MyDrive/SCv3_compressed/val.zip" -d "/content/local/"
!unzip -q "/content/drive/MyDrive/SCv3_compressed/test.zip" -d "/content/local/"

!echo "Unzipping train set"
!unzip -q "/content/drive/MyDrive/SCv3_compressed/train-001.zip" -d "/content/local/"
!unzip -q "/content/drive/MyDrive/SCv3_compressed/train-002.zip" -d "/content/local/"
!unzip -q "/content/drive/MyDrive/SCv3_compressed/train-003.zip" -d "/content/local/"
!unzip -q "/content/drive/MyDrive/SCv3_compressed/train-004.zip" -d "/content/local/"

!echo "LOAD COMPLETE"

Unzipping test & val sets
Unzipping train set
LOAD COMPLETE


In [43]:
"""
validate local val, train, test sets
"""

base_dir = '/content/local'
folders = {'train': 40000, 'val': 5000, 'test': 5000}

for folder, expected_count in folders.items():
    folder_path = os.path.join(base_dir, folder)

    if os.path.exists(folder_path):
        count = len([f for f in os.listdir(folder_path) if f.endswith('.parquet')])
        print(f"{folder:5} | {count:,} files, expected {expected_count:,} files.")
    else:
        print(f"{folder:5} | No Directory")

Checking dataset integrity in /content/local...

train | 40,000 files, expected 40,000 files.
val   | 5,000 files, expected 5,000 files.
test  | 5,000 files, expected 5,000 files.


In [44]:
class Local_Loader(Dataset):
    def __init__(self, dir_dataset="/content/local", phase=None, num_samples_low=655, num_samples_high=6553, sample_size=None):

        if phase is None:
            raise ValueError("Phase cannot be None.")
        self.dir_dataset = os.path.join(dir_dataset, phase)

        self.pqt_files = sorted(glob.glob(os.path.join(self.dir_dataset, "*.parquet")))

        if sample_size is not None:
            self.pqt_files = self.pqt_files[:sample_size]

        self.num_samples_low = num_samples_low
        self.num_samples_high = num_samples_high

        self.width = 256
        self.height = 256

    def __len__(self):
        return len(self.pqt_files)

    def __getitem__(self, idx):
        """
        loads data per sample_idx
        """
        pqt_name = self.pqt_files[idx]
        df = pd.read_parquet(pqt_name, engine='pyarrow')

        # reshape channels
        env_map = df['building_mask'].iloc[0].reshape((self.height, self.width))
        tx_map = df['tx_origin'].iloc[0].reshape((self.height, self.width))
        radio_map = df['path_loss'].iloc[0].reshape((self.height, self.width))

        # create sparsity (around 1% to 10% of originial radio map)
        num_samples = np.random.randint(self.num_samples_low, self.num_samples_high)
        sparse_samples = np.zeros((self.height, self.width), dtype=np.float32)

        x_samples = np.random.randint(0, self.width, size=num_samples)
        y_samples = np.random.randint(0, self.height, size=num_samples)
        sparse_samples[x_samples, y_samples] = radio_map[x_samples, y_samples]

        MIN_DB = 30.0
        MAX_DB = 140.0

        # Normalize map
        radio_map_norm = (radio_map - MIN_DB) / (MAX_DB - MIN_DB)

        sparse_samples_norm = np.zeros_like(sparse_samples)
        valid_mask = sparse_samples > 0
        sparse_samples_norm[valid_mask] = (sparse_samples[valid_mask] - MIN_DB) / (MAX_DB - MIN_DB)
        sparse_samples_norm = np.clip(sparse_samples_norm, 0.0, 1.0)

        # stack tensor
        inputs_np = np.stack([env_map, tx_map, sparse_samples_norm], axis=0)
        target_np = np.expand_dims(radio_map_norm, axis=0)

        return torch.from_numpy(inputs_np).float(), torch.from_numpy(target_np).float()

In [45]:
# load data
Radio_train = Local_Loader(phase="train", sample_size=25000)
Radio_val = Local_Loader(phase="val", sample_size=2000)
Radio_test = Local_Loader(phase="test", sample_size=2000)

image_datasets = {
    'train': Radio_train,
    'val': Radio_val,
    'test': Radio_test
}

BATCH_SIZE = 128
NUM_WORKERS = 8

dataloaders = {
    'train': DataLoader(Radio_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2),
    'val': DataLoader(Radio_val, batch_size=BATCH_SIZE, shuffle=True, num_workers=2),
    'test': DataLoader(Radio_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
}

In [46]:
def convrelu(in_channels, out_channels, kernel, padding, pool):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel, padding=padding),
        #In conv, the dimension of the output, if the input is H,W, is
        # H+2*padding-kernel +1
        nn.ReLU(inplace=True),
        nn.MaxPool2d(pool, stride=pool, padding=0, dilation=1, return_indices=False, ceil_mode=False)
        #pooling takes Height H and width W to (H-pool)/pool+1 = H/pool, and floor. Same for W.
        #altogether, the output size is (H+2*padding-kernel +1)/pool.
    )

def convreluT(in_channels, out_channels, kernel, padding):
    return nn.Sequential(
        nn.ConvTranspose2d(in_channels, out_channels, kernel, stride=2, padding=padding),
        nn.ReLU(inplace=True)
        #input is H X W, output is   (H-1)*2 - 2*padding + kernel
    )

# RadioUnet module (W for to U-nets)
class RadioWnet(nn.Module):
    def __init__(self,inputs=2,phase="firstU"):
        super().__init__()

        self.inputs=inputs
        self.phase=phase

        #---1st U-net---
        #------------------Encoder--------------------
        if inputs<=3:
            self.layer00 = convrelu(inputs, 6, 3, 1,1)
            self.layer0 = convrelu(6, 40, 5, 2,2)
        else:
            self.layer00 = convrelu(inputs, 10, 3, 1,1)
            self.layer0 = convrelu(10, 40, 5, 2,2)

        self.layer1 = convrelu(40, 50, 5, 2,2)
        self.layer10 = convrelu(50, 60, 5, 2,1)
        self.layer2 = convrelu(60, 100, 5, 2,2)
        self.layer20 = convrelu(100, 100, 3, 1,1)
        self.layer3 = convrelu(100, 150, 5, 2,2)
        self.layer4 =convrelu(150, 300, 5, 2,2)
        self.layer5 =convrelu(300, 500, 5, 2,2)
        #------------------Decoder--------------------
        self.conv_up5 =convreluT(500, 300, 4, 1)
        self.conv_up4 = convreluT(300+300, 150, 4, 1)
        self.conv_up3 = convreluT(150 + 150, 100, 4, 1)
        self.conv_up20 = convrelu(100 + 100, 100, 3, 1, 1)
        self.conv_up2 = convreluT(100 + 100, 60, 6, 2)
        self.conv_up10 = convrelu(60 + 60, 50, 5, 2, 1)
        self.conv_up1 = convreluT(50 + 50, 40, 6, 2)
        self.conv_up0 = convreluT(40 + 40, 20, 6, 2)
        if inputs<=3:
            self.conv_up00 = convrelu(20+6+inputs, 20, 5, 2,1)

        else:
            self.conv_up00 = convrelu(20+10+inputs, 20, 5, 2,1)

        self.conv_up000 = convrelu(20+inputs, 1, 5, 2,1)
        #---2nd U-net---
        self.Wlayer00 = convrelu(inputs+1, 20, 3, 1,1)
        self.Wlayer0 = convrelu(20, 30, 5, 2,2)
        self.Wlayer1 = convrelu(30, 40, 5, 2,2)
        self.Wlayer10 = convrelu(40, 50, 5, 2,1)
        self.Wlayer2 = convrelu(50, 60, 5, 2,2)
        self.Wlayer20 = convrelu(60, 70, 3, 1,1)
        self.Wlayer3 = convrelu(70, 90, 5, 2,2)
        self.Wlayer4 =convrelu(90, 110, 5, 2,2)
        self.Wlayer5 =convrelu(110, 150, 5, 2,2)

        self.Wconv_up5 =convreluT(150, 110, 4, 1)
        self.Wconv_up4 = convreluT(110+110, 90, 4, 1)
        self.Wconv_up3 = convreluT(90 + 90, 70, 4, 1)
        self.Wconv_up20 = convrelu(70 + 70, 60, 3, 1, 1)
        self.Wconv_up2 = convreluT(60 + 60, 50, 6, 2)
        self.Wconv_up10 = convrelu(50 + 50, 40, 5, 2, 1)
        self.Wconv_up1 = convreluT(40 + 40, 30, 6, 2)
        self.Wconv_up0 = convreluT(30 + 30, 20, 6, 2)
        self.Wconv_up00 = convrelu(20+20+inputs+1, 20, 5, 2,1)
        self.Wconv_up000 = convrelu(20+inputs+1, 1, 5, 2,1)

    def forward(self, input):

        input0=input[:,0:self.inputs,:,:]

        if self.phase=="firstU":
            layer00 = self.layer00(input0)
            layer0 = self.layer0(layer00)
            layer1 = self.layer1(layer0)
            layer10 = self.layer10(layer1)
            layer2 = self.layer2(layer10)
            layer20 = self.layer20(layer2)
            layer3 = self.layer3(layer20)
            layer4 = self.layer4(layer3)
            layer5 = self.layer5(layer4)

            layer4u = self.conv_up5(layer5)
            layer4u = torch.cat([layer4u, layer4], dim=1)
            layer3u = self.conv_up4(layer4u)
            layer3u = torch.cat([layer3u, layer3], dim=1)
            layer20u = self.conv_up3(layer3u)
            layer20u = torch.cat([layer20u, layer20], dim=1)
            layer2u = self.conv_up20(layer20u)
            layer2u = torch.cat([layer2u, layer2], dim=1)
            layer10u = self.conv_up2(layer2u)
            layer10u = torch.cat([layer10u, layer10], dim=1)
            layer1u = self.conv_up10(layer10u)
            layer1u = torch.cat([layer1u, layer1], dim=1)
            layer0u = self.conv_up1(layer1u)
            layer0u = torch.cat([layer0u, layer0], dim=1)
            layer00u = self.conv_up0(layer0u)
            layer00u = torch.cat([layer00u, layer00], dim=1)
            layer00u = torch.cat([layer00u,input0], dim=1)
            layer000u  = self.conv_up00(layer00u)
            layer000u = torch.cat([layer000u,input0], dim=1)
            output1  = self.conv_up000(layer000u)

            Winput=torch.cat([output1, input], dim=1).detach()

            Wlayer00 = self.Wlayer00(Winput).detach()
            Wlayer0 = self.Wlayer0(Wlayer00).detach()
            Wlayer1 = self.Wlayer1(Wlayer0).detach()
            Wlayer10 = self.Wlayer10(Wlayer1).detach()
            Wlayer2 = self.Wlayer2(Wlayer10).detach()
            Wlayer20 = self.Wlayer20(Wlayer2).detach()
            Wlayer3 = self.Wlayer3(Wlayer20).detach()
            Wlayer4 = self.Wlayer4(Wlayer3).detach()
            Wlayer5 = self.Wlayer5(Wlayer4).detach()

            Wlayer4u = self.Wconv_up5(Wlayer5).detach()
            Wlayer4u = torch.cat([Wlayer4u, Wlayer4], dim=1).detach()
            Wlayer3u = self.Wconv_up4(Wlayer4u).detach()
            Wlayer3u = torch.cat([Wlayer3u, Wlayer3], dim=1).detach()
            Wlayer20u = self.Wconv_up3(Wlayer3u).detach()
            Wlayer20u = torch.cat([Wlayer20u, Wlayer20], dim=1).detach()
            Wlayer2u = self.Wconv_up20(Wlayer20u).detach()
            Wlayer2u = torch.cat([Wlayer2u, Wlayer2], dim=1).detach()
            Wlayer10u = self.Wconv_up2(Wlayer2u).detach()
            Wlayer10u = torch.cat([Wlayer10u, Wlayer10], dim=1).detach()
            Wlayer1u = self.Wconv_up10(Wlayer10u).detach()
            Wlayer1u = torch.cat([Wlayer1u, Wlayer1], dim=1).detach()
            Wlayer0u = self.Wconv_up1(Wlayer1u).detach()
            Wlayer0u = torch.cat([Wlayer0u, Wlayer0], dim=1).detach()
            Wlayer00u = self.Wconv_up0(Wlayer0u).detach()
            Wlayer00u = torch.cat([Wlayer00u, Wlayer00], dim=1).detach()
            Wlayer00u = torch.cat([Wlayer00u,Winput], dim=1).detach()
            Wlayer000u  = self.Wconv_up00(Wlayer00u).detach()
            Wlayer000u = torch.cat([Wlayer000u,Winput], dim=1).detach()
            output2  = self.Wconv_up000(Wlayer000u).detach()

        else:
            layer00 = self.layer00(input0).detach()
            layer0 = self.layer0(layer00).detach()
            layer1 = self.layer1(layer0).detach()
            layer10 = self.layer10(layer1).detach()
            layer2 = self.layer2(layer10).detach()
            layer20 = self.layer20(layer2).detach()
            layer3 = self.layer3(layer20).detach()
            layer4 = self.layer4(layer3).detach()
            layer5 = self.layer5(layer4).detach()

            layer4u = self.conv_up5(layer5).detach()
            layer4u = torch.cat([layer4u, layer4], dim=1).detach()
            layer3u = self.conv_up4(layer4u).detach()
            layer3u = torch.cat([layer3u, layer3], dim=1).detach()
            layer20u = self.conv_up3(layer3u).detach()
            layer20u = torch.cat([layer20u, layer20], dim=1).detach()
            layer2u = self.conv_up20(layer20u).detach()
            layer2u = torch.cat([layer2u, layer2], dim=1).detach()
            layer10u = self.conv_up2(layer2u).detach()
            layer10u = torch.cat([layer10u, layer10], dim=1).detach()
            layer1u = self.conv_up10(layer10u).detach()
            layer1u = torch.cat([layer1u, layer1], dim=1).detach()
            layer0u = self.conv_up1(layer1u).detach()
            layer0u = torch.cat([layer0u, layer0], dim=1).detach()
            layer00u = self.conv_up0(layer0u).detach()
            layer00u = torch.cat([layer00u, layer00], dim=1).detach()
            layer00u = torch.cat([layer00u,input0], dim=1).detach()
            layer000u  = self.conv_up00(layer00u).detach()
            layer000u = torch.cat([layer000u,input0], dim=1).detach()
            output1  = self.conv_up000(layer000u).detach()

            Winput=torch.cat([output1, input], dim=1).detach()

            Wlayer00 = self.Wlayer00(Winput)
            Wlayer0 = self.Wlayer0(Wlayer00)
            Wlayer1 = self.Wlayer1(Wlayer0)
            Wlayer10 = self.Wlayer10(Wlayer1)
            Wlayer2 = self.Wlayer2(Wlayer10)
            Wlayer20 = self.Wlayer20(Wlayer2)
            Wlayer3 = self.Wlayer3(Wlayer20)
            Wlayer4 = self.Wlayer4(Wlayer3)
            Wlayer5 = self.Wlayer5(Wlayer4)

            Wlayer4u = self.Wconv_up5(Wlayer5)
            Wlayer4u = torch.cat([Wlayer4u, Wlayer4], dim=1)
            Wlayer3u = self.Wconv_up4(Wlayer4u)
            Wlayer3u = torch.cat([Wlayer3u, Wlayer3], dim=1)
            Wlayer20u = self.Wconv_up3(Wlayer3u)
            Wlayer20u = torch.cat([Wlayer20u, Wlayer20], dim=1)
            Wlayer2u = self.Wconv_up20(Wlayer20u)
            Wlayer2u = torch.cat([Wlayer2u, Wlayer2], dim=1)
            Wlayer10u = self.Wconv_up2(Wlayer2u)
            Wlayer10u = torch.cat([Wlayer10u, Wlayer10], dim=1)
            Wlayer1u = self.Wconv_up10(Wlayer10u)
            Wlayer1u = torch.cat([Wlayer1u, Wlayer1], dim=1)
            Wlayer0u = self.Wconv_up1(Wlayer1u)
            Wlayer0u = torch.cat([Wlayer0u, Wlayer0], dim=1)
            Wlayer00u = self.Wconv_up0(Wlayer0u)
            Wlayer00u = torch.cat([Wlayer00u, Wlayer00], dim=1)
            Wlayer00u = torch.cat([Wlayer00u,Winput], dim=1)
            Wlayer000u  = self.Wconv_up00(Wlayer00u)
            Wlayer000u = torch.cat([Wlayer000u,Winput], dim=1)
            output2  = self.Wconv_up000(Wlayer000u)

        return [output1,output2]

In [47]:
from torchsummary import summary

torch.set_default_dtype(torch.float32)
# torch.set_default_tensor_type('torch.cuda.FloatTensor')
torch.backends.cudnn.enabled
model=RadioWnet(inputs=3, phase="firstU")
model.cuda()
summary(model, input_size=(3,256,256))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1          [-1, 6, 256, 256]             168
              ReLU-2          [-1, 6, 256, 256]               0
         MaxPool2d-3          [-1, 6, 256, 256]               0
            Conv2d-4         [-1, 40, 256, 256]           6,040
              ReLU-5         [-1, 40, 256, 256]               0
         MaxPool2d-6         [-1, 40, 128, 128]               0
            Conv2d-7         [-1, 50, 128, 128]          50,050
              ReLU-8         [-1, 50, 128, 128]               0
         MaxPool2d-9           [-1, 50, 64, 64]               0
           Conv2d-10           [-1, 60, 64, 64]          75,060
             ReLU-11           [-1, 60, 64, 64]               0
        MaxPool2d-12           [-1, 60, 64, 64]               0
           Conv2d-13          [-1, 100, 64, 64]         150,100
             ReLU-14          [-1, 100,

Training Loop

In [48]:
import torch
import torch.optim as optim
from torch.optim import lr_scheduler
import time
import copy
from collections import defaultdict
import torch.nn.functional as F
import torch.nn as nn

def calc_loss_dense(pred, target, metrics):
    criterion = nn.MSELoss()
    loss = criterion(pred, target)
    metrics['loss'] += loss.data.cpu().numpy() * target.size(0)

    return loss

def calc_loss_sparse(pred, target, samples, metrics, num_samples):
    criterion = nn.MSELoss()
    loss = criterion(samples*pred, samples*target)*(256**2)/num_samples
    metrics['loss'] += loss.data.cpu().numpy() * target.size(0)

    return loss

def print_metrics(metrics, epoch_samples, phase):
    outputs1 = []
    outputs2 = []
    for k in metrics.keys():
        outputs1.append("{}: {:4f}".format(k, metrics[k] / (epoch_samples*256**2)))

    print("{}: {}".format(phase, ", ".join(outputs1)))

def train_model(model, optimizer, scheduler, num_epochs=50, WNetPhase="firstU", targetType="dense", num_samples=300):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = 1e10

    training_history = {
        'epoch': [],
        'train_u1': [], 'val_u1': [],
        'train_u2': [], 'val_u2': []
    }

    csv_save_path = '/content/wnet_training_history.csv'

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        since = time.time()

        # MSE logging variables
        epoch_tracker_data = {'train_u1': 0.0, 'train_u2': 0.0, 'val_u1': 0.0, 'val_u2': 0.0}

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                scheduler.step()
                for param_group in optimizer.param_groups:
                    print("learning rate", param_group['lr'])
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            metrics = defaultdict(float)
            epoch_samples = 0

            if targetType == "dense":
                for inputs, targets in dataloaders[phase]:
                    inputs = inputs.to(device)
                    targets = targets.to(device)

                    optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == 'train'):
                        [outputs1, outputs2] = model(inputs)

                        # --- track loss per epoch
                        # We calculate simple MSE for both outputs just for the graph
                        epoch_tracker_data[f'{phase}_u1'] += F.mse_loss(outputs1, targets).item() * inputs.size(0)
                        epoch_tracker_data[f'{phase}_u2'] += F.mse_loss(outputs2, targets).item() * inputs.size(0)

                        # loss calculation for the optimizer
                        if WNetPhase == "firstU":
                            loss = calc_loss_dense(outputs1, targets, metrics)
                        else:
                            loss = calc_loss_dense(outputs2, targets, metrics)

                        if phase == 'train':
                            loss.backward()
                            optimizer.step()

                    epoch_samples += inputs.size(0)

            elif targetType == "sparse":
                for inputs, targets, samples in dataloaders[phase]:
                    inputs = inputs.to(device)
                    targets = targets.to(device)
                    samples = samples.to(device)

                    optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == 'train'):
                        [outputs1, outputs2] = model(inputs)

                        # track loss per epoch
                        epoch_tracker_data[f'{phase}_u1'] += F.mse_loss(outputs1, targets).item() * inputs.size(0)
                        epoch_tracker_data[f'{phase}_u2'] += F.mse_loss(outputs2, targets).item() * inputs.size(0)

                        if WNetPhase == "firstU":
                            loss = calc_loss_sparse(outputs1, targets, samples, metrics, num_samples)
                        else:
                            loss = calc_loss_sparse(outputs2, targets, samples, metrics, num_samples)

                        if phase == 'train':
                            loss.backward()
                            optimizer.step()

                    epoch_samples += inputs.size(0)

            print_metrics(metrics, epoch_samples, phase)
            epoch_loss = metrics['loss'] / epoch_samples

            # Finalize tracking averages for this phase
            epoch_tracker_data[f'{phase}_u1'] /= epoch_samples
            epoch_tracker_data[f'{phase}_u2'] /= epoch_samples

            # deep copy the model
            if phase == 'val' and epoch_loss < best_loss:
                print("saving best model")
                best_loss = epoch_loss
                best_model_wts = copy.deepcopy(model.state_dict())

        # This executes once both the 'train' and 'val' loops are finished for the current epoch
        training_history['epoch'].append(epoch)
        training_history['train_u1'].append(epoch_tracker_data['train_u1'])
        training_history['val_u1'].append(epoch_tracker_data['val_u1'])
        training_history['train_u2'].append(epoch_tracker_data['train_u2'])
        training_history['val_u2'].append(epoch_tracker_data['val_u2'])

        # Immediately overwrite the CSV
        df_history = pd.DataFrame(training_history)
        df_history.to_csv(csv_save_path, index=False)

        time_elapsed = time.time() - since
        print('{:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))

    model.load_state_dict(best_model_wts)
    return model

Training First U-net

In [49]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

optimizer_ft = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=30, gamma=0.1)

model = train_model(model, optimizer_ft, exp_lr_scheduler, WNetPhase="firstU")

cuda:0
Epoch 0/3
----------
learning rate 0.0001
train: loss: 0.000004
val: loss: 0.000001
saving best model
2m 22s
Epoch 1/3
----------
learning rate 0.0001
train: loss: 0.000001
val: loss: 0.000001
saving best model
2m 25s
Epoch 2/3
----------
learning rate 0.0001
train: loss: 0.000001
val: loss: 0.000000
saving best model
2m 25s
Epoch 3/3
----------
learning rate 0.0001
train: loss: 0.000001
val: loss: 0.000001
2m 25s


Saving & loading first U-net Model

In [50]:
# saving 1st U-net
try:
    os.mkdir('RadioWNet')
except OSError as error:
    print(error)

torch.save(model.state_dict(), 'RadioWNet/Trained_Model_FirstU.pt')

[Errno 17] File exists: 'RadioWNet'


In [51]:
# load state
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RadioWnet(inputs=3, phase="firstU")
model.load_state_dict(torch.load('RadioWNet/Trained_Model_FirstU.pt'))
model.to(device)

RadioWnet(
  (layer00): Sequential(
    (0): Conv2d(3, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=1, stride=1, padding=0, dilation=1, ceil_mode=False)
  )
  (layer0): Sequential(
    (0): Conv2d(6, 40, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): Conv2d(40, 50, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer10): Sequential(
    (0): Conv2d(50, 60, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=1, stride=1, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(60, 100, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU(inplace=True)
  

In [52]:
# load 2nd U-net
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RadioWnet(inputs=3, phase="secondU")
model.load_state_dict(torch.load('RadioWNet/Trained_Model_FirstU.pt'))
model.to(device)

RadioWnet(
  (layer00): Sequential(
    (0): Conv2d(3, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=1, stride=1, padding=0, dilation=1, ceil_mode=False)
  )
  (layer0): Sequential(
    (0): Conv2d(6, 40, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): Conv2d(40, 50, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer10): Sequential(
    (0): Conv2d(50, 60, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=1, stride=1, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(60, 100, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU(inplace=True)
  

Train 2nd U-Net

In [53]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

optimizer_ft = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=30, gamma=0.1)

model = train_model(model, optimizer_ft, exp_lr_scheduler, WNetPhase="secondU")

cuda:0
Epoch 0/3
----------
learning rate 0.0001
train: loss: 0.000003
val: loss: 0.000001
saving best model
2m 24s
Epoch 1/3
----------
learning rate 0.0001
train: loss: 0.000001
val: loss: 0.000001
saving best model
2m 23s
Epoch 2/3
----------
learning rate 0.0001
train: loss: 0.000000
val: loss: 0.000000
saving best model
2m 24s
Epoch 3/3
----------
learning rate 0.0001
train: loss: 0.000000
val: loss: 0.000000
saving best model
2m 24s


Saving and Loading 2nd U-net model

In [54]:
torch.save(model.state_dict(), 'RadioWNet/Trained_Model_SecondU.pt')

In [55]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RadioWnet(inputs=3, phase="secondU")
model.load_state_dict(torch.load('RadioWNet/Trained_Model_SecondU.pt'))
model.to(device)

RadioWnet(
  (layer00): Sequential(
    (0): Conv2d(3, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=1, stride=1, padding=0, dilation=1, ceil_mode=False)
  )
  (layer0): Sequential(
    (0): Conv2d(6, 40, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): Conv2d(40, 50, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer10): Sequential(
    (0): Conv2d(50, 60, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=1, stride=1, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(60, 100, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU(inplace=True)
  

Test accuracy

In [56]:
import torch
import torch.optim as optim
from torch.optim import lr_scheduler
import time
import copy
from collections import defaultdict
import torch.nn.functional as F
import torch.nn as nn

def calc_loss_test(pred1, pred2, target, metrics, error="MSE"):
    """
    Calculates standard MSE or Normalized MSE for two sequential predictions.
    Accumulates the batch loss into the metrics dictionary.
    """
    criterion = nn.MSELoss()

    if error == "MSE":
        loss1 = criterion(pred1, target)
        loss2 = criterion(pred2, target)
    else:
        # NMSE
        target_power = (target ** 2).mean()
        loss1 = criterion(pred1, target) / target_power
        loss2 = criterion(pred2, target) / target_power

    metrics['loss first U'] += loss1.item() * target.size(0)
    metrics['loss second U'] += loss2.item() * target.size(0)

    return [loss1, loss2]

Visualization

In [1]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def visualize_predictions(model, dataloader, device, num_samples=1, min_db=30.0, max_db=140.0):
    """
    Evaluates the U-Net architecture.
    Plots training loss and reconstruction visualization
    """
    model.eval()

    # Loss/Epoch plot
    csv_path = '/content/wnet_training_history.csv'

    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        if not df.empty:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4))

            ax1.plot(df['epoch'], df['train_u1'], label='Train U1', color='blue')
            if 'val_u1' in df.columns and not df['val_u1'].isnull().all():
                ax1.plot(df['epoch'], df['val_u1'], label='Val U1', color='lightblue', linestyle='--')
            ax1.set_title('1st U-Net (Coarse) MSE')
            ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss (Normalized)'); ax1.legend(); ax1.grid(True, alpha=0.3)

            ax2.plot(df['epoch'], df['train_u2'], label='Train U2', color='red')
            if 'val_u2' in df.columns and not df['val_u2'].isnull().all():
                ax2.plot(df['epoch'], df['val_u2'], label='Val U2', color='salmon', linestyle='--')
            ax2.set_title('2nd U-Net (Refined) MSE')
            ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss (Normalized)'); ax2.legend(); ax2.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()
    else:
        print(f"No loss history found at {csv_path}")

    # forward pass
    inputs, targets = next(iter(dataloader))
    inputs = inputs.to(device)
    targets = targets.to(device)

    with torch.no_grad():
        outputs1, outputs2 = model(inputs)

    def unnormalize(tensor):
        return tensor.cpu().numpy() * (max_db - min_db) + min_db

    inputs_np = inputs.cpu().numpy()
    targets_db = unnormalize(targets)
    out1_db = unnormalize(outputs1)
    out2_db = unnormalize(outputs2)

    # Sparse channel un-normalized
    sparse_db = inputs_np[:, 2, :, :].copy()
    sparse_db[sparse_db > 0] = sparse_db[sparse_db > 0] * (max_db - min_db) + min_db

    # Flatten arrays for statistical plotting
    flat_true = targets_db.flatten()
    flat_pred = out2_db.flatten()
    residuals = flat_pred - flat_true
    rmse = np.sqrt(np.mean(residuals**2))

    for i in range(min(num_samples, inputs.size(0))):
        fig, axarr = plt.subplots(2, 4, figsize=(20, 10))

        # Calculate Absolute Errors
        err1 = np.abs(out1_db[i, 0] - targets_db[i, 0])
        err2 = np.abs(out2_db[i, 0] - targets_db[i, 0])
        vmax_err = max(np.max(err1), np.max(err2))

        # Input visualized
        axarr[0, 0].imshow(inputs_np[i, 0], cmap='gray')
        axarr[0, 0].set_title('Input 0: Buildings'); axarr[0, 0].axis('off')

        axarr[0, 1].imshow(inputs_np[i, 1], cmap='hot')
        axarr[0, 1].set_title('Input 1: Tx Location'); axarr[0, 1].axis('off')

        # Sparse Sensors visualized
        axarr[0, 2].imshow(inputs_np[i, 0], cmap='gray', alpha=0.3)
        sparse_masked = np.ma.masked_where(sparse_db[i] == 0, sparse_db[i])
        im_sparse = axarr[0, 2].imshow(sparse_masked, cmap='jet', vmin=min_db, vmax=max_db)
        pct_coverage = (np.count_nonzero(sparse_db[i]) / (256*256)) * 100
        axarr[0, 2].set_title(f'Input 2: Sensors ({pct_coverage:.2f}% known)')
        axarr[0, 2].axis('off')

        im_gt = axarr[0, 3].imshow(targets_db[i, 0], cmap='jet', vmin=min_db, vmax=max_db)
        axarr[0, 3].set_title('Target: Ground Truth')
        axarr[0, 3].axis('off'); plt.colorbar(im_gt, ax=axarr[0, 3], fraction=0.046, pad=0.04)

        # Predictions vizualized
        im_o1 = axarr[1, 0].imshow(out1_db[i, 0], cmap='jet', vmin=min_db, vmax=max_db)
        axarr[1, 0].set_title('U-Net 1 Prediction')
        axarr[1, 0].axis('off'); plt.colorbar(im_o1, ax=axarr[1, 0], fraction=0.046, pad=0.04)

        im_o2 = axarr[1, 2].imshow(out2_db[i, 0], cmap='jet', vmin=min_db, vmax=max_db)
        axarr[1, 2].set_title('U-Net 2 Refinement')
        axarr[1, 2].axis('off'); plt.colorbar(im_o2, ax=axarr[1, 2], fraction=0.046, pad=0.04)

        plt.tight_layout()
        plt.show()

visualize_predictions(model, dataloaders['val'], device)

NameError: name 'model' is not defined